In [3]:
%pip install numpy pandas openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# ============================================================
# AFM GITHUB PAGES DASHBOARD GENERATOR
# ============================================================
#
# This creates:
#
# index.html
#
# data/
#     manifest.json
#     plus_ve_cell1part1_xxxxxxxx.json
#     plus_ve_cell1part2_xxxxxxxx.json
#     minus_ve_cell1part1_xxxxxxxx.json
#     20_cell1part1_xxxxxxxx.json
#     320_cell1part1_xxxxxxxx.json
#     ...
#
# The HTML page loads only the selected folder.
#
# This keeps index.html small and makes the dashboard suitable
# for GitHub Pages.
#
# AFM folders should look like:
#
# afm_data/
#     +ve_cell1part1/
#     +ve_cell1part2/
#     -ve_cell1part1/
#     20_cell1part1/
#     320_cell1part1/
#
# Excel files should look like:
#
# afm_values/
#     +ve.xlsx
#     -ve.xlsx
#     20.xlsx
#     320.xlsx
#
# Example:
#
# AFM folder:
# +ve_cell2part1
#
# maps to:
# +ve.xlsx
#
# sheet:
# cell2part1
#
# ============================================================


# ============================================================
# IMPORT PACKAGES
# ============================================================

from pathlib import Path
import hashlib
import json
import re

import numpy as np
import pandas as pd


# ============================================================
# MAIN FOLDERS
# ============================================================

AFM_FOLDER = Path("afm_data")

EXCEL_FOLDER = Path("afm_values")

DATA_FOLDER = Path("data")

INDEX_FILE = Path("index.html")

MANIFEST_FILE = DATA_FOLDER / "manifest.json"


# ============================================================
# DEFAULT GRAPH RANGE
# ============================================================

DEFAULT_X_MIN = -150

DEFAULT_X_MAX = 200


# ============================================================
# FIT SETTINGS
# ============================================================
#
# Fit regions are selected along the force (y) axis.
#
# Linear fit:
#     y >= 0.2 nN, up to the maximum force of the curve.
#
# Elasticity fit:
#     y < 0.2 nN, with no lower force cutoff.
#
# ============================================================

FORCE_BOUNDARY_NN = 0.2

POISSON_RATIO = 0.50

PYRAMID_HALF_ANGLE_DEG = 20.0

MIN_FIT_POINTS = 20


# ============================================================
# CURVE NUMBER OFFSET
# ============================================================
#
# Normally:
#
# cropped_032.txt -> Excel Curve 32
#
# Leave this as 0.
#
# If your Excel numbering is shifted, for example:
#
# cropped_001.txt -> Excel Curve 0
#
# change this to:
#
# CURVE_OFFSET = -1
#
# ============================================================

CURVE_OFFSET = 0


# ============================================================
# CREATE DATA FOLDER
# ============================================================

DATA_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# REMOVE OLD GENERATED JSON FILES
# ============================================================
#
# This prevents old folders remaining in the website after
# you regenerate the dashboard.
#
# ============================================================

for old_json in DATA_FOLDER.glob("*.json"):

    old_json.unlink()


# ============================================================
# NATURAL SORTING
# ============================================================
#
# This makes:
#
# cell1part2
# cell1part10
#
# sort numerically rather than alphabetically.
#
# ============================================================

def natural_sort_key(text):

    return [

        int(part)
        if part.isdigit()
        else part.lower()

        for part in re.split(
            r"(\d+)",
            str(text)
        )
    ]


# ============================================================
# READ ONE AFM TEXT FILE
# ============================================================

def load_afm_txt(file_path):

    segments = {}

    current_segment = None

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        for line in file:

            line = line.strip()


            # -----------------------------------------------
            # Detect approach or retract section
            # -----------------------------------------------

            if line.startswith("# segment:"):

                current_segment = (
                    line
                    .split(":", 1)[1]
                    .strip()
                    .lower()
                )

                segments.setdefault(
                    current_segment,
                    []
                )

                continue


            # -----------------------------------------------
            # Ignore blank lines
            # -----------------------------------------------

            if not line:

                continue


            # -----------------------------------------------
            # Ignore header lines
            # -----------------------------------------------

            if line.startswith("#"):

                continue


            # -----------------------------------------------
            # Ignore data before first segment
            # -----------------------------------------------

            if current_segment is None:

                continue


            # -----------------------------------------------
            # Convert row to numbers
            # -----------------------------------------------

            try:

                row = [

                    float(value)

                    for value
                    in line.split()

                ]

            except ValueError:

                continue


            # -----------------------------------------------
            # Store row
            # -----------------------------------------------

            if len(row) >= 2:

                segments[
                    current_segment
                ].append(row)


    # --------------------------------------------------------
    # Convert lists into NumPy arrays
    # --------------------------------------------------------

    approach = np.asarray(
        segments.get(
            "extend",
            []
        ),
        dtype=float
    )

    retract = np.asarray(
        segments.get(
            "retract",
            []
        ),
        dtype=float
    )


    return approach, retract


# ============================================================
# READ ONE VALUE FROM AN AFM TXT HEADER
# ============================================================

def read_header_value(
    file_path,
    key
):

    prefix = f"# {key}:"

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="ignore"
    ) as file:

        for line in file:

            if line.startswith(prefix):

                return (
                    line
                    .split(":", 1)[1]
                    .strip()
                )

    return None


# ============================================================
# EXTRACT CURVE NUMBER FROM FILE NAME
# ============================================================
#
# Example:
#
# qi-data-xxxx-cropped_157.txt
#
# returns:
#
# 157
#
# ============================================================

def extract_curve_number(file_path):

    filename = file_path.name


    # Preferred method
    match = re.search(

        r"cropped_(\d+)\.txt$",

        filename,

        re.IGNORECASE
    )


    if match:

        return (
            int(match.group(1))
            +
            CURVE_OFFSET
        )


    # Backup method
    match = re.search(

        r"_(\d+)\.txt$",

        filename,

        re.IGNORECASE
    )


    if match:

        return (
            int(match.group(1))
            +
            CURVE_OFFSET
        )


    return None


# ============================================================
# CONVERT EXCEL CURVE VALUE INTO INTEGER
# ============================================================
#
# This handles:
#
# 1
# 1.0
# _001
# 001
#
# ============================================================

def excel_curve_to_number(value):

    if pd.isna(value):

        return None


    # --------------------------------------------------------
    # Numeric Excel value
    # --------------------------------------------------------

    if isinstance(
        value,
        (int, float, np.integer, np.floating)
    ):

        try:

            if np.isfinite(value):

                return int(value)

        except Exception:

            pass


    # --------------------------------------------------------
    # String Excel value
    # --------------------------------------------------------

    text = str(value).strip()


    match = re.search(
        r"(\d+)$",
        text
    )


    if match:

        return int(
            match.group(1)
        )


    return None


# ============================================================
# SAFE NUMBER CONVERSION
# ============================================================
#
# Converts valid Excel numbers to float.
#
# #VALUE!, blanks and other invalid cells become None.
#
# ============================================================

def safe_number(value):

    try:

        number = float(value)

        if np.isfinite(number):

            return number

    except Exception:

        pass


    return None


# ============================================================
# EXTRACT CURVE NUMBER FROM A NAME OR TSV FILENAME FIELD
# ============================================================

def extract_curve_number_from_name(name):

    text = str(name)

    match = re.search(
        r"cropped_(\d+)",
        text,
        re.IGNORECASE
    )

    if match:

        return (
            int(match.group(1))
            + CURVE_OFFSET
        )

    match = re.search(
        r"_(\d+)(?:\.[A-Za-z0-9]+)?$",
        text,
        re.IGNORECASE
    )

    if match:

        return (
            int(match.group(1))
            + CURVE_OFFSET
        )

    return None


# ============================================================
# CALCULATE SAMPLE STIFFNESS kB
# ============================================================

def calculate_kb_nN_per_um(
    spring_constant_N_per_m,
    slope_N_per_m
):

    if (
        spring_constant_N_per_m is None
        or slope_N_per_m is None
    ):

        return None

    denominator = (
        spring_constant_N_per_m
        + slope_N_per_m
    )

    if denominator == 0:

        return None

    kb_N_per_m = (
        spring_constant_N_per_m
        * slope_N_per_m
        / denominator
    )

    return (
        kb_N_per_m
        * 1000.0
    )


# ============================================================
# JPK PROCESSED TSV CACHE
# ============================================================

processed_tsv_cache = {}

all_processed_tsv_files_cache = None


# ============================================================
# READ ONE JPK PROCESSED TSV
# ============================================================

def read_processed_tsv(tsv_file):

    cache_key = str(tsv_file.resolve())

    if cache_key in processed_tsv_cache:

        return processed_tsv_cache[cache_key]

    try:

        dataframe = pd.read_csv(
            tsv_file,
            sep="\t"
        )

    except Exception as error:

        result = (
            None,
            f"Could not read {tsv_file.name}: {error}"
        )

        processed_tsv_cache[cache_key] = result

        return result

    required = [
        "Filename",
        "Slope [N/m]",
        "Young's Modulus [Pa]",
        "Contact Point [m]",
        "Baseline [N]",
        "ResidualRMS [N]"
    ]

    missing = [
        column
        for column in required
        if column not in dataframe.columns
    ]

    if missing:

        result = (
            None,
            f"Missing TSV columns in {tsv_file.name}: {missing}"
        )

        processed_tsv_cache[cache_key] = result

        return result

    result = (
        dataframe,
        "OK"
    )

    processed_tsv_cache[cache_key] = result

    return result


# ============================================================
# FIND JPK PROCESSED TSV FILES FOR ONE AFM FOLDER
# ============================================================

def find_processed_tsv_files(
    folder,
    curve_files
):

    local_files = sorted(
        folder.glob("*processed*.tsv")
    )

    if local_files:

        return local_files

    acquisition_prefixes = {
        file_path.name.split(
            "-cropped_",
            1
        )[0]
        for file_path in curve_files
        if "-cropped_" in file_path.name
    }

    if not acquisition_prefixes:

        return []

    global all_processed_tsv_files_cache

    if all_processed_tsv_files_cache is None:

        all_processed_tsv_files_cache = list(
            AFM_FOLDER.rglob(
                "*processed*.tsv"
            )
        )

    matches = [
        path
        for path in all_processed_tsv_files_cache
        if any(
            path.name.startswith(prefix)
            for prefix in acquisition_prefixes
        )
    ]

    return sorted(matches)


# ============================================================
# BUILD JPK VALUE MAP FOR ONE AFM FOLDER
# ============================================================

def build_jpk_map(
    folder,
    curve_files
):

    tsv_files = find_processed_tsv_files(
        folder,
        curve_files
    )

    metadata = {
        "tsv_files": [
            path.name
            for path in tsv_files
        ],
        "jpk_status": "OK"
    }

    if not tsv_files:

        metadata["jpk_status"] = (
            "No processed JPK TSV found"
        )

        return {}, metadata

    jpk_map = {}

    errors = []

    for tsv_file in tsv_files:

        dataframe, status = (
            read_processed_tsv(
                tsv_file
            )
        )

        if dataframe is None:

            errors.append(status)

            continue

        for _, row in dataframe.iterrows():

            curve_number = (
                extract_curve_number_from_name(
                    row["Filename"]
                )
            )

            if curve_number is None:

                continue

            slope = safe_number(
                row["Slope [N/m]"]
            )

            youngs_modulus = safe_number(
                row["Young's Modulus [Pa]"]
            )

            contact = safe_number(
                row["Contact Point [m]"]
            )

            baseline = safe_number(
                row["Baseline [N]"]
            )

            residual_rms = safe_number(
                row["ResidualRMS [N]"]
            )

            if slope is not None:

                slope = abs(slope)

            jpk_map[curve_number] = {
                "slope_N_per_m": slope,
                "youngs_modulus_Pa": youngs_modulus,
                "contact_m": contact,
                "baseline_N": baseline,
                "residual_rms_N": residual_rms,
                "tsv_file": tsv_file.name
            }

    if not jpk_map:

        metadata["jpk_status"] = (
            "; ".join(errors)
            if errors
            else "No curve rows found in processed JPK TSV"
        )

    elif errors:

        metadata["jpk_status"] = (
            "Loaded with warnings: "
            + "; ".join(errors)
        )

    return jpk_map, metadata


# ============================================================
# FIT ONE CURVE USING THE JPK COORDINATE SYSTEM
# ============================================================
#
# The JPK contact point and baseline define the elasticity model.
# The x shift is inferred by reproducing the JPK saved elasticity
# residual with the JPK saved modulus.
#
# After the shift is found:
#
# Linear fit:
#     all finite approach points with y >= 0.2 nN.
#
# Elasticity fit:
#     all finite approach points with y < 0.2 nN.
#     There is no lower y-force cutoff.
#
# The fitted elasticity coefficient is solved by least squares,
# with the JPK contact point and baseline held fixed.
#
# ============================================================

def fit_curve_to_jpk(
    approach,
    retract,
    spring_constant_N_per_m,
    jpk_values
):

    fit_settings = {
        "force_boundary_nN": FORCE_BOUNDARY_NN,
        "poisson_ratio": POISSON_RATIO,
        "pyramid_half_angle_deg": PYRAMID_HALF_ANGLE_DEG
    }

    if approach.size == 0:

        return {
            **fit_settings,
            "status": "No Extend segment found"
        }

    if spring_constant_N_per_m is None:

        return {
            **fit_settings,
            "status": "springConstant missing from TXT"
        }

    if not jpk_values:

        return {
            **fit_settings,
            "status": "No matching JPK TSV row"
        }

    required_values = {
        "slope_N_per_m": jpk_values.get(
            "slope_N_per_m"
        ),
        "youngs_modulus_Pa": jpk_values.get(
            "youngs_modulus_Pa"
        ),
        "contact_m": jpk_values.get(
            "contact_m"
        ),
        "baseline_N": jpk_values.get(
            "baseline_N"
        ),
        "residual_rms_N": jpk_values.get(
            "residual_rms_N"
        )
    }

    missing = [
        key
        for key, value in required_values.items()
        if value is None
    ]

    if missing:

        return {
            **fit_settings,
            "status": (
                "Missing JPK values: "
                + ", ".join(missing)
            )
        }

    x_m = approach[:, 0].astype(float)

    y_N = approach[:, 1].astype(float)

    finite = (
        np.isfinite(x_m)
        & np.isfinite(y_N)
    )

    force_boundary_N = (
        FORCE_BOUNDARY_NN
        * 1e-9
    )

    linear_mask = (
        finite
        & (y_N >= force_boundary_N)
    )

    elasticity_mask = (
        finite
        & (y_N < force_boundary_N)
    )

    linear_count = int(
        np.count_nonzero(
            linear_mask
        )
    )

    elasticity_count = int(
        np.count_nonzero(
            elasticity_mask
        )
    )

    if linear_count < MIN_FIT_POINTS:

        return {
            **fit_settings,
            "status": (
                f"Only {linear_count} points in linear region"
            )
        }

    if elasticity_count < MIN_FIT_POINTS:

        return {
            **fit_settings,
            "status": (
                f"Only {elasticity_count} points in elasticity region"
            )
        }

    slope_jpk = required_values[
        "slope_N_per_m"
    ]

    E_jpk_Pa = required_values[
        "youngs_modulus_Pa"
    ]

    contact_m = required_values[
        "contact_m"
    ]

    baseline_N = required_values[
        "baseline_N"
    ]

    rms_jpk_N = required_values[
        "residual_rms_N"
    ]

    nu = POISSON_RATIO

    alpha_rad = np.deg2rad(
        PYRAMID_HALF_ANGLE_DEG
    )

    geometry_factor = (
        0.7453
        / (1 - nu**2)
        * np.tan(alpha_rad)
    )

    if geometry_factor == 0:

        return {
            **fit_settings,
            "status": "Elasticity geometry factor is zero"
        }

    jpk_elasticity_coefficient = (
        geometry_factor
        * E_jpk_Pa
    )

    # --------------------------------------------------------
    # Linear slope does not change with an x translation.
    # Compute it once for the shift scoring.
    # --------------------------------------------------------

    preliminary_slope, _ = np.polyfit(
        x_m[linear_mask],
        y_N[linear_mask],
        1
    )

    preliminary_slope = abs(
        float(preliminary_slope)
    )

    if slope_jpk == 0:

        slope_relative_error = 0.0

    else:

        slope_relative_error = (
            abs(
                preliminary_slope
                - slope_jpk
            )
            / abs(slope_jpk)
        )

    def evaluate_shift(shift_nm):

        x_processed_m = (
            x_m
            + shift_nm * 1e-9
        )

        delta_m = np.maximum(
            contact_m
            - x_processed_m[
                elasticity_mask
            ],
            0.0
        )

        predicted_N = (
            baseline_N
            + jpk_elasticity_coefficient
            * delta_m**2
        )

        reconstructed_rms_N = float(
            np.sqrt(
                np.mean(
                    (
                        y_N[elasticity_mask]
                        - predicted_N
                    )**2
                )
            )
        )

        if rms_jpk_N == 0:

            rms_relative_error = (
                reconstructed_rms_N
            )

        else:

            rms_relative_error = (
                abs(
                    reconstructed_rms_N
                    - rms_jpk_N
                )
                / abs(rms_jpk_N)
            )

        score = (
            slope_relative_error**2
            + rms_relative_error**2
        )

        return (
            score,
            reconstructed_rms_N
        )

    # --------------------------------------------------------
    # Coarse x-shift search
    # --------------------------------------------------------

    coarse_candidates = []

    for shift_nm in np.arange(
        -300.0,
        300.01,
        0.5
    ):

        score, reconstructed_rms = (
            evaluate_shift(
                shift_nm
            )
        )

        coarse_candidates.append(
            (
                score,
                abs(shift_nm),
                shift_nm,
                reconstructed_rms
            )
        )

    coarse_candidates.sort(
        key=lambda item: (
            item[0],
            item[1]
        )
    )

    best_coarse_shift = (
        coarse_candidates[0][2]
    )

    # --------------------------------------------------------
    # Fine x-shift search
    # --------------------------------------------------------

    fine_candidates = []

    for shift_nm in np.arange(
        best_coarse_shift - 1.0,
        best_coarse_shift + 1.001,
        0.01
    ):

        score, reconstructed_rms = (
            evaluate_shift(
                shift_nm
            )
        )

        fine_candidates.append(
            (
                score,
                abs(shift_nm),
                shift_nm,
                reconstructed_rms
            )
        )

    fine_candidates.sort(
        key=lambda item: (
            item[0],
            item[1]
        )
    )

    best_shift_nm = float(
        fine_candidates[0][2]
    )

    x_processed_m = (
        x_m
        + best_shift_nm * 1e-9
    )

    # --------------------------------------------------------
    # Final linear fit
    # --------------------------------------------------------

    linear_slope, linear_intercept = (
        np.polyfit(
            x_processed_m[linear_mask],
            y_N[linear_mask],
            1
        )
    )

    fitted_slope_N_per_m = abs(
        float(linear_slope)
    )

    fitted_intercept_N = float(
        linear_intercept
    )

    linear_prediction = (
        linear_slope
        * x_processed_m[linear_mask]
        + linear_intercept
    )

    ss_res = float(
        np.sum(
            (
                y_N[linear_mask]
                - linear_prediction
            )**2
        )
    )

    centered = (
        y_N[linear_mask]
        - np.mean(
            y_N[linear_mask]
        )
    )

    ss_tot = float(
        np.sum(
            centered**2
        )
    )

    if ss_tot == 0:

        linear_r2 = None

    else:

        linear_r2 = (
            1.0
            - ss_res / ss_tot
        )

    # --------------------------------------------------------
    # Fit elasticity coefficient and E by least squares.
    # Contact point and baseline remain fixed at JPK values.
    # --------------------------------------------------------

    delta_fit_m = np.maximum(
        contact_m
        - x_processed_m[elasticity_mask],
        0.0
    )

    delta_squared = (
        delta_fit_m**2
    )

    elasticity_target = (
        y_N[elasticity_mask]
        - baseline_N
    )

    denominator = float(
        np.dot(
            delta_squared,
            delta_squared
        )
    )

    if denominator <= 0:

        return {
            **fit_settings,
            "status": (
                "Elasticity region has no indentation "
                "before the contact point"
            )
        }

    fitted_elasticity_coefficient = float(
        np.dot(
            delta_squared,
            elasticity_target
        )
        / denominator
    )

    fitted_elasticity_coefficient = max(
        fitted_elasticity_coefficient,
        0.0
    )

    fitted_E_Pa = (
        fitted_elasticity_coefficient
        / geometry_factor
    )

    fitted_elasticity_prediction_N = (
        baseline_N
        + fitted_elasticity_coefficient
        * delta_squared
    )

    fitted_rms_N = float(
        np.sqrt(
            np.mean(
                (
                    y_N[elasticity_mask]
                    - fitted_elasticity_prediction_N
                )**2
            )
        )
    )

    fitted_kb_nN_per_um = (
        calculate_kb_nN_per_um(
            spring_constant_N_per_m,
            fitted_slope_N_per_m
        )
    )

    # --------------------------------------------------------
    # Build visible fit lines.
    # --------------------------------------------------------

    linear_x_m = np.linspace(
        np.min(
            x_processed_m[linear_mask]
        ),
        np.max(
            x_processed_m[linear_mask]
        ),
        180
    )

    linear_y_N = (
        linear_slope
        * linear_x_m
        + linear_intercept
    )

    elasticity_start_m = float(
        np.min(
            x_processed_m[elasticity_mask]
        )
    )

    elasticity_end_m = float(
        contact_m
    )

    if elasticity_start_m < elasticity_end_m:

        elasticity_x_m = np.linspace(
            elasticity_start_m,
            elasticity_end_m,
            220
        )

        elasticity_delta_m = np.maximum(
            contact_m
            - elasticity_x_m,
            0.0
        )

        elasticity_y_N = (
            baseline_N
            + fitted_elasticity_coefficient
            * elasticity_delta_m**2
        )

    else:

        elasticity_x_m = np.asarray(
            [],
            dtype=float
        )

        elasticity_y_N = np.asarray(
            [],
            dtype=float
        )

    approach_x_processed_nm = (
        x_processed_m
        * 1e9
    )

    if retract.size == 0:

        retract_x_processed_nm = np.asarray(
            [],
            dtype=float
        )

    else:

        retract_x_processed_nm = (
            retract[:, 0].astype(float)
            * 1e9
            + best_shift_nm
        )

    return {
        **fit_settings,
        "status": "OK",
        "x_shift_nm": best_shift_nm,
        "linear_point_count": linear_count,
        "elasticity_point_count": elasticity_count,
        "s_N_per_m": fitted_slope_N_per_m,
        "kB_nN_per_um": fitted_kb_nN_per_um,
        "E_MPa": (
            fitted_E_Pa
            / 1e6
        ),
        "RMS_pN": (
            fitted_rms_N
            * 1e12
        ),
        "linear_intercept_pN": (
            fitted_intercept_N
            * 1e12
        ),
        "linear_r2": (
            float(linear_r2)
            if linear_r2 is not None
            else None
        ),
        "approach_x_processed_nm": np.round(
            approach_x_processed_nm,
            6
        ).tolist(),
        "retract_x_processed_nm": np.round(
            retract_x_processed_nm,
            6
        ).tolist(),
        "linear_x_nm": np.round(
            linear_x_m * 1e9,
            6
        ).tolist(),
        "linear_y_nN": np.round(
            linear_y_N * 1e9,
            6
        ).tolist(),
        "elasticity_x_nm": np.round(
            elasticity_x_m * 1e9,
            6
        ).tolist(),
        "elasticity_y_nN": np.round(
            elasticity_y_N * 1e9,
            6
        ).tolist()
    }


# ============================================================
# NORMALISE EXCEL COLUMN NAMES
# ============================================================

def normalise_column_name(name):

    return (
        str(name)
        .strip()
        .lower()
        .replace("μ", "µ")
        .replace(" ", "")
    )


# ============================================================
# FIND EXCEL COLUMN
# ============================================================

def find_column(
    dataframe,
    possible_names
):

    lookup = {

        normalise_column_name(column):
            column

        for column
        in dataframe.columns
    }


    for possible_name in possible_names:

        normalised = (
            normalise_column_name(
                possible_name
            )
        )

        if normalised in lookup:

            return lookup[
                normalised
            ]


    return None


# ============================================================
# EXCEL CACHE
# ============================================================
#
# Sheets are read only once.
#
# ============================================================

excel_sheet_cache = {}


# ============================================================
# READ ONE EXCEL SHEET
# ============================================================
#
# This automatically ignores spaces before or after sheet
# names.
#
# Example:
#
# "cell2part1 "
#
# will match:
#
# "cell2part1"
#
# ============================================================

def read_excel_sheet(
    excel_file,
    requested_sheet
):

    cache_key = (

        str(excel_file.resolve()),

        requested_sheet.strip()
    )


    if cache_key in excel_sheet_cache:

        return (
            excel_sheet_cache[
                cache_key
            ],
            "OK"
        )


    # --------------------------------------------------------
    # Check Excel file exists
    # --------------------------------------------------------

    if not excel_file.exists():

        return (
            None,
            f"Excel file not found: {excel_file}"
        )


    try:

        excel_book = pd.ExcelFile(
            excel_file
        )

    except Exception as error:

        return (
            None,
            f"Could not open Excel: {error}"
        )


    # --------------------------------------------------------
    # Clean sheet names
    # --------------------------------------------------------

    sheet_lookup = {

        sheet.strip():
            sheet

        for sheet
        in excel_book.sheet_names
    }


    clean_requested_sheet = (
        requested_sheet.strip()
    )


    # --------------------------------------------------------
    # Check sheet exists
    # --------------------------------------------------------

    if clean_requested_sheet not in sheet_lookup:

        return (
            None,
            (
                f"Sheet '{clean_requested_sheet}' "
                f"not found in {excel_file.name}"
            )
        )


    # --------------------------------------------------------
    # Get actual Excel sheet name
    # --------------------------------------------------------

    actual_sheet = (
        sheet_lookup[
            clean_requested_sheet
        ]
    )


    try:

        dataframe = pd.read_excel(

            excel_file,

            sheet_name=actual_sheet

        )

    except Exception as error:

        return (
            None,
            f"Could not read sheet: {error}"
        )


    # --------------------------------------------------------
    # Clean column names
    # --------------------------------------------------------

    dataframe.columns = [

        str(column).strip()

        for column
        in dataframe.columns
    ]


    excel_sheet_cache[
        cache_key
    ] = dataframe


    return (
        dataframe,
        "OK"
    )


# ============================================================
# DETERMINE CONDITION AND SHEET FROM AFM FOLDER
# ============================================================
#
# Examples:
#
# +ve_cell1part2
#
# condition = +ve
# sheet     = cell1part2
#
#
# 20_cell2part1
#
# condition = 20
# sheet     = cell2part1
#
# ============================================================

def get_folder_mapping(folder):

    folder_name = folder.name


    if "_" not in folder_name:

        return (
            None,
            None,
            None
        )


    condition, sheet_name = (

        folder_name.split(
            "_",
            1
        )
    )


    condition = (
        condition.strip()
    )

    sheet_name = (
        sheet_name.strip()
    )


    excel_file = (

        EXCEL_FOLDER

        /

        f"{condition}.xlsx"
    )


    return (
        condition,
        sheet_name,
        excel_file
    )


# ============================================================
# CREATE VALUE MAP FOR ONE AFM FOLDER
# ============================================================
#
# The result looks like:
#
# {
#     1: {
#         "kB": ...,
#         "E": ...,
#         "RMS": ...
#     },
#
#     2: {
#         ...
#     }
# }
#
# ============================================================

def build_value_map(folder):

    (
        condition,
        sheet_name,
        excel_file

    ) = get_folder_mapping(
        folder
    )


    metadata = {

        "condition":
            condition,

        "sheet":
            sheet_name,

        "excel_file":
            (
                excel_file.name
                if excel_file
                else None
            ),

        "excel_status":
            "OK"
    }


    if excel_file is None:

        metadata["excel_status"] = (
            "Could not determine Excel mapping"
        )

        return {}, metadata


    dataframe, status = (
        read_excel_sheet(

            excel_file,

            sheet_name
        )
    )


    metadata[
        "excel_status"
    ] = status


    if dataframe is None:

        return {}, metadata


    # --------------------------------------------------------
    # Find columns
    # --------------------------------------------------------

    curve_column = find_column(

        dataframe,

        [
            "Curve"
        ]
    )


    kb_column = find_column(

        dataframe,

        [
            "kB (nN/µm)",
            "kB (nN/μm)",
            "kB"
        ]
    )


    e_column = find_column(

        dataframe,

        [
            "E (MPa)",
            "E"
        ]
    )


    rms_column = find_column(

        dataframe,

        [
            "RMS (pN)",
            "RMS"
        ]
    )


    # --------------------------------------------------------
    # Curve column is required
    # --------------------------------------------------------

    if curve_column is None:

        metadata[
            "excel_status"
        ] = "Curve column not found"

        return {}, metadata


    # --------------------------------------------------------
    # Build lookup table
    # --------------------------------------------------------

    value_map = {}


    for _, row in dataframe.iterrows():

        curve_number = (
            excel_curve_to_number(
                row[
                    curve_column
                ]
            )
        )


        if curve_number is None:

            continue


        value_map[
            curve_number
        ] = {

            "kB":
                (
                    safe_number(
                        row[kb_column]
                    )
                    if kb_column
                    else None
                ),

            "E":
                (
                    safe_number(
                        row[e_column]
                    )
                    if e_column
                    else None
                ),

            "RMS":
                (
                    safe_number(
                        row[rms_column]
                    )
                    if rms_column
                    else None
                )
        }


    return (
        value_map,
        metadata
    )


# ============================================================
# CLEAN AND CONVERT AFM ARRAY
# ============================================================
#
# Converts:
#
# metres -> nm
# Newtons -> nN
#
# Values are rounded to 6 decimal places in the dashboard JSON
# to keep file sizes smaller.
#
# The original AFM text files are not modified.
#
# ============================================================

def convert_segment_to_nm_nN(
    segment
):

    if segment.size == 0:

        return [], []


    x = (
        segment[:, 0]
        *
        1e9
    )


    y = (
        segment[:, 1]
        *
        1e9
    )


    # --------------------------------------------------------
    # Remove invalid points
    # --------------------------------------------------------

    valid = (

        np.isfinite(x)

        &

        np.isfinite(y)
    )


    x = x[valid]

    y = y[valid]


    # --------------------------------------------------------
    # Round data
    # --------------------------------------------------------

    x = np.round(
        x,
        6
    )

    y = np.round(
        y,
        6
    )


    return (
        x.tolist(),
        y.tolist()
    )


# ============================================================
# PROCESS ONE AFM CURVE
# ============================================================

def process_curve(
    file_path,
    value_map,
    jpk_map
):

    approach, retract = (
        load_afm_txt(
            file_path
        )
    )

    curve_number = (
        extract_curve_number(
            file_path
        )
    )

    # --------------------------------------------------------
    # Convert AFM data
    # --------------------------------------------------------

    approach_x, approach_y = (
        convert_segment_to_nm_nN(
            approach
        )
    )

    retract_x, retract_y = (
        convert_segment_to_nm_nN(
            retract
        )
    )

    # --------------------------------------------------------
    # Find matching Excel values
    # --------------------------------------------------------

    values = value_map.get(
        curve_number,
        {
            "kB": None,
            "E": None,
            "RMS": None
        }
    )

    # --------------------------------------------------------
    # Find matching JPK values
    # --------------------------------------------------------

    jpk_values = jpk_map.get(
        curve_number
    )

    # --------------------------------------------------------
    # Read cantilever spring constant from AFM TXT
    # --------------------------------------------------------

    spring_constant_N_per_m = safe_number(
        read_header_value(
            file_path,
            "springConstant"
        )
    )

    # --------------------------------------------------------
    # Build Actual (JPK) values
    # --------------------------------------------------------

    actual_jpk = {
        "kC_N_per_m": spring_constant_N_per_m,
        "s_N_per_m": None,
        "kB_nN_per_um": None,
        "E_MPa": None,
        "RMS_pN": None,
        "contact_nm": None,
        "baseline_pN": None,
        "tsv_file": None
    }

    if jpk_values:

        slope_jpk = jpk_values.get(
            "slope_N_per_m"
        )

        E_jpk_Pa = jpk_values.get(
            "youngs_modulus_Pa"
        )

        rms_jpk_N = jpk_values.get(
            "residual_rms_N"
        )

        contact_m = jpk_values.get(
            "contact_m"
        )

        baseline_N = jpk_values.get(
            "baseline_N"
        )

        actual_jpk.update({
            "s_N_per_m": slope_jpk,
            "kB_nN_per_um": (
                calculate_kb_nN_per_um(
                    spring_constant_N_per_m,
                    slope_jpk
                )
            ),
            "E_MPa": (
                E_jpk_Pa / 1e6
                if E_jpk_Pa is not None
                else None
            ),
            "RMS_pN": (
                rms_jpk_N * 1e12
                if rms_jpk_N is not None
                else None
            ),
            "contact_nm": (
                contact_m * 1e9
                if contact_m is not None
                else None
            ),
            "baseline_pN": (
                baseline_N * 1e12
                if baseline_N is not None
                else None
            ),
            "tsv_file": jpk_values.get(
                "tsv_file"
            )
        })

    # --------------------------------------------------------
    # Fit curve
    # --------------------------------------------------------

    fit = fit_curve_to_jpk(
        approach,
        retract,
        spring_constant_N_per_m,
        jpk_values
    )

    # --------------------------------------------------------
    # Use the inferred JPK x coordinate for plotting when fit
    # reconstruction succeeds.
    # --------------------------------------------------------

    if fit.get("status") == "OK":

        approach_x = fit.get(
            "approach_x_processed_nm",
            approach_x
        )

        retract_x = fit.get(
            "retract_x_processed_nm",
            retract_x
        )

    if curve_number is None:

        curve_label = (
            file_path.stem
        )

    else:

        curve_label = (
            f"_{curve_number:03d}"
        )

    # --------------------------------------------------------
    # Excel match status
    # --------------------------------------------------------

    if curve_number is None:

        match_status = (
            "Curve number not found in filename"
        )

    elif curve_number in value_map:

        match_status = "OK"

    else:

        match_status = (
            "Curve not found in Excel sheet"
        )

    return {
        "filename": file_path.name,
        "curve_number": curve_number,
        "curve_label": curve_label,
        "kB": values["kB"],
        "E": values["E"],
        "RMS": values["RMS"],
        "match_status": match_status,
        "jpk": actual_jpk,
        "fit": fit,
        "approach_x": approach_x,
        "approach_y": approach_y,
        "retract_x": retract_x,
        "retract_y": retract_y
    }


# ============================================================
# CREATE SAFE JSON FILE NAME
# ============================================================
#
# A hash is included so different folder names cannot
# accidentally create the same filename.
#
# ============================================================

def make_json_filename(
    folder_name
):

    slug = (

        folder_name

        .replace(
            "+",
            "plus_"
        )

        .replace(
            "-",
            "minus_"
        )
    )


    slug = re.sub(

        r"[^A-Za-z0-9._]+",

        "_",

        slug
    )


    digest = (

        hashlib
        .sha1(
            folder_name.encode(
                "utf-8"
            )
        )
        .hexdigest()[:8]
    )


    return (
        f"{slug}_{digest}.json"
    )


# ============================================================
# FIND ALL AFM FOLDERS
# ============================================================

folder_paths = {

    file.parent

    for file
    in AFM_FOLDER.rglob("*.txt")

}


folder_paths = sorted(

    folder_paths,

    key=lambda folder:
        natural_sort_key(
            folder.relative_to(
                AFM_FOLDER
            )
        )
)


# ============================================================
# STOP IF NO AFM DATA EXISTS
# ============================================================

if not folder_paths:

    raise FileNotFoundError(

        f"No .txt files found inside "
        f"{AFM_FOLDER.resolve()}"

    )


# ============================================================
# PROCESS ALL FOLDERS
# ============================================================

manifest = []

total_curves = 0


for folder in folder_paths:


    # --------------------------------------------------------
    # Relative folder name
    # --------------------------------------------------------

    folder_name = str(

        folder.relative_to(
            AFM_FOLDER
        )
    )


    # --------------------------------------------------------
    # Find AFM curves
    # --------------------------------------------------------

    files = list(
        folder.glob("*.txt")
    )


    # --------------------------------------------------------
    # Last curve first
    #
    # Example:
    #
    # _157
    # _156
    # _155
    # ...
    #
    # --------------------------------------------------------

    files = sorted(

        files,

        key=lambda file: (

            extract_curve_number(file)

            if extract_curve_number(file)
            is not None

            else -1
        ),

        reverse=True
    )


    # --------------------------------------------------------
    # Read matching Excel sheet once
    # --------------------------------------------------------

    value_map, metadata = (

        build_value_map(
            folder
        )
    )


    jpk_map, jpk_metadata = (

        build_jpk_map(
            folder,
            files
        )
    )


    print()

    print(
        f"Processing: "
        f"{folder_name} "
        f"({len(files)} curves)"
    )


    print(
        "Excel:",
        metadata.get(
            "excel_file"
        )
    )


    print(
        "Sheet:",
        metadata.get(
            "sheet"
        )
    )


    print(
        "Excel status:",
        metadata.get(
            "excel_status"
        )
    )


    print(
        "JPK TSV status:",
        jpk_metadata.get(
            "jpk_status"
        )
    )


    print(
        "JPK TSV files:",
        jpk_metadata.get(
            "tsv_files"
        )
    )


    # --------------------------------------------------------
    # Process curves
    # --------------------------------------------------------

    curves = []


    for file_path in files:

        curve_data = (

            process_curve(

                file_path,

                value_map,

                jpk_map
            )
        )


        curves.append(
            curve_data
        )


    total_curves += len(
        curves
    )


    # --------------------------------------------------------
    # Build folder JSON
    # --------------------------------------------------------

    folder_data = {

        "folder":
            folder_name,

        "condition":
            metadata.get(
                "condition"
            ),

        "sheet":
            metadata.get(
                "sheet"
            ),

        "excel_file":
            metadata.get(
                "excel_file"
            ),

        "excel_status":
            metadata.get(
                "excel_status"
            ),

        "jpk_status":
            jpk_metadata.get(
                "jpk_status"
            ),

        "jpk_tsv_files":
            jpk_metadata.get(
                "tsv_files"
            ),

        "curves":
            curves
    }


    # --------------------------------------------------------
    # Create JSON filename
    # --------------------------------------------------------

    json_filename = (

        make_json_filename(
            folder_name
        )
    )


    json_path = (

        DATA_FOLDER
        /
        json_filename
    )


    # --------------------------------------------------------
    # Save folder JSON
    # --------------------------------------------------------

    with open(

        json_path,

        "w",

        encoding="utf-8"

    ) as json_file:

        json.dump(

            folder_data,

            json_file,

            separators=(
                ",",
                ":"
            ),

            ensure_ascii=False,

            allow_nan=False
        )


    # --------------------------------------------------------
    # Add folder to manifest
    # --------------------------------------------------------

    manifest.append({

        "folder":
            folder_name,

        "file":
            json_filename,

        "count":
            len(curves),

        "condition":
            metadata.get(
                "condition"
            ),

        "sheet":
            metadata.get(
                "sheet"
            )
    })


# ============================================================
# SAVE MANIFEST
# ============================================================

with open(

    MANIFEST_FILE,

    "w",

    encoding="utf-8"

) as manifest_file:

    json.dump(

        manifest,

        manifest_file,

        indent=2,

        ensure_ascii=False,

        allow_nan=False
    )


# ============================================================
# CREATE INDEX.HTML
# ============================================================
#
# Plotly is loaded from its CDN.
#
# The dashboard then loads JSON files from the data folder.
#
# ============================================================

html = r"""
<!DOCTYPE html>

<html lang="en">

<head>

<meta charset="UTF-8">

<meta
    name="viewport"
    content="width=device-width, initial-scale=1.0"
>

<title>AFM Dashboard</title>


<script
    src="https://cdn.plot.ly/plotly-2.35.2.min.js">
</script>


<style>

body {
    margin: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #f4f5f7;
    color: #202124;
}


header {
    background: white;
    border-bottom: 1px solid #dddddd;
    padding: 18px 28px;
}


header h1 {
    margin: 0;
    font-size: 24px;
}


header p {
    margin: 6px 0 0 0;
    color: #666666;
}


.controls {
    background: white;
    margin: 20px;
    padding: 18px;
    border-radius: 8px;
}


.control-row {
    display: flex;
    gap: 12px;
    flex-wrap: wrap;
    align-items: center;
    margin-bottom: 12px;
}


label {
    font-weight: 600;
}


select,
input,
button {
    font-size: 14px;
    padding: 8px 10px;
}


select {
    min-width: 300px;
}


button {
    cursor: pointer;
}


.main-layout {
    display: grid;
    grid-template-columns: minmax(600px, 1fr) 390px;
    gap: 20px;
    margin: 20px;
}


.card {
    background: white;
    border-radius: 8px;
    padding: 16px;
}


#plot {
    width: 100%;
    height: 620px;
}


.value-card h2 {
    margin-top: 0;
}


.parameter {
    margin-bottom: 24px;
}


.parameter-name {
    color: #666666;
    font-size: 14px;
}


.parameter-value {
    font-size: 24px;
    margin-top: 4px;
}


.parameter-unit {
    color: #555555;
    margin-top: 2px;
}


.fit-section {
    border-top: 1px solid #e2e2e2;
    padding-top: 14px;
    margin-top: 14px;
}


.fit-section:first-of-type {
    border-top: 0;
    padding-top: 0;
    margin-top: 0;
}


.fit-section h3 {
    margin: 0 0 10px 0;
    font-size: 16px;
}


.metric-row {
    display: grid;
    grid-template-columns: minmax(118px, 1fr) auto auto;
    gap: 8px;
    align-items: baseline;
    margin: 6px 0;
    font-size: 14px;
}


.metric-label {
    color: #555555;
}


.metric-value {
    font-family: Consolas, Monaco, monospace;
    text-align: right;
}


.metric-unit {
    color: #666666;
    min-width: 52px;
}


.fit-setting {
    font-size: 13px;
    line-height: 1.5;
    margin: 5px 0;
}


.fit-status {
    margin-top: 10px;
    font-size: 13px;
}


.metadata {
    margin-top: 25px;
    font-size: 13px;
    line-height: 1.6;
    overflow-wrap: anywhere;
}


.status-ok {
    color: #18794e;
}


.status-error {
    color: #b3261e;
}


.table-card {
    margin: 20px;
}


.table-wrapper {
    overflow-x: auto;
    max-height: 600px;
    overflow-y: auto;
}


table {
    width: 100%;
    border-collapse: collapse;
}


th,
td {
    padding: 9px 12px;
    border-bottom: 1px solid #e5e5e5;
    text-align: left;
}


th {
    position: sticky;
    top: 0;
    background: white;
    z-index: 2;
}


tbody tr {
    cursor: pointer;
}


tbody tr:hover {
    background: #f1f3f4;
}


tbody tr.selected {
    background: #e8f0fe;
}


#loading {
    color: #555555;
    margin-left: 10px;
}


.counter {
    color: #555555;
}


@media (max-width: 900px) {

    .main-layout {
        grid-template-columns: 1fr;
    }

    #plot {
        height: 500px;
    }
}

</style>

</head>


<body>


<header>

    <h1>
        AFM Force Indentation Dashboard
    </h1>

    <p>
        Approach, retract and mechanical parameters
    </p>

</header>


<div class="controls">


    <div class="control-row">

        <label for="folderSelect">
            Folder
        </label>

        <select id="folderSelect"></select>

        <span id="loading"></span>

    </div>


    <div class="control-row">

        <label for="curveSelect">
            Curve
        </label>

        <select id="curveSelect"></select>

    </div>


    <div class="control-row">

        <button id="previousButton">
            Previous
        </button>

        <button id="nextButton">
            Next
        </button>

        <span
            class="counter"
            id="curveCounter">
        </span>

    </div>


    <div class="control-row">

        <label for="xMin">
            X min (nm)
        </label>

        <input
            id="xMin"
            type="number"
            value="__X_MIN__"
        >

        <label for="xMax">
            X max (nm)
        </label>

        <input
            id="xMax"
            type="number"
            value="__X_MAX__"
        >

        <button id="applyAxis">
            Apply range
        </button>

    </div>


</div>


<div class="main-layout">


    <div class="card">

        <div id="plot"></div>

    </div>


    <div class="card value-card">

        <h2>
            Curve values
        </h2>


        <div class="fit-section">

            <h3>Actual (JPK)</h3>

            <div class="metric-row">
                <span class="metric-label">kC</span>
                <span class="metric-value" id="jpkKcValue">N/A</span>
                <span class="metric-unit">N/m</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">s</span>
                <span class="metric-value" id="jpkSlopeValue">N/A</span>
                <span class="metric-unit">N/m</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">kB</span>
                <span class="metric-value" id="jpkKbValue">N/A</span>
                <span class="metric-unit">nN/µm</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">E</span>
                <span class="metric-value" id="jpkEValue">N/A</span>
                <span class="metric-unit">MPa</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">RMS</span>
                <span class="metric-value" id="jpkRmsValue">N/A</span>
                <span class="metric-unit">pN</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">Contact point</span>
                <span class="metric-value" id="jpkContactValue">N/A</span>
                <span class="metric-unit">nm</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">Baseline</span>
                <span class="metric-value" id="jpkBaselineValue">N/A</span>
                <span class="metric-unit">pN</span>
            </div>

        </div>


        <div class="fit-section">

            <h3>From fitting</h3>

            <div class="metric-row">
                <span class="metric-label">s</span>
                <span class="metric-value" id="fitSlopeValue">N/A</span>
                <span class="metric-unit">N/m</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">kB</span>
                <span class="metric-value" id="fitKbValue">N/A</span>
                <span class="metric-unit">nN/µm</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">E</span>
                <span class="metric-value" id="fitEValue">N/A</span>
                <span class="metric-unit">MPa</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">RMS</span>
                <span class="metric-value" id="fitRmsValue">N/A</span>
                <span class="metric-unit">pN</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">Linear intercept</span>
                <span class="metric-value" id="fitInterceptValue">N/A</span>
                <span class="metric-unit">pN</span>
            </div>

            <div class="fit-status">
                <strong>Fit:</strong>
                <span id="fitStatusValue">N/A</span>
            </div>

        </div>


        <div class="fit-section">

            <h3>Fit settings</h3>

            <div class="fit-setting" id="fitLinearSetting">
                Linear: N/A
            </div>

            <div class="fit-setting" id="fitElasticitySetting">
                Elasticity: N/A
            </div>

            <div class="fit-setting">
                ν = <span id="fitNuValue">N/A</span>,
                α = <span id="fitAlphaValue">N/A</span>°
            </div>

        </div>


        <div class="fit-section">

            <h3>Excel reference</h3>

            <div class="metric-row">
                <span class="metric-label">kB</span>
                <span class="metric-value" id="kbValue">N/A</span>
                <span class="metric-unit">nN/µm</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">E</span>
                <span class="metric-value" id="eValue">N/A</span>
                <span class="metric-unit">MPa</span>
            </div>

            <div class="metric-row">
                <span class="metric-label">RMS</span>
                <span class="metric-value" id="rmsValue">N/A</span>
                <span class="metric-unit">pN</span>
            </div>

        </div>


        <div class="metadata">

            <div>
                <strong>Curve:</strong>
                <span id="curveValue"></span>
            </div>

            <div>
                <strong>Folder:</strong>
                <span id="folderValue"></span>
            </div>

            <div>
                <strong>Excel:</strong>
                <span id="excelValue"></span>
            </div>

            <div>
                <strong>Sheet:</strong>
                <span id="sheetValue"></span>
            </div>

            <div>
                <strong>JPK TSV:</strong>
                <span id="jpkTsvValue"></span>
            </div>

            <div>
                <strong>File:</strong>
                <span id="filenameValue"></span>
            </div>

            <div>
                <strong>Excel match:</strong>
                <span id="matchValue"></span>
            </div>

        </div>

    </div>


</div>


<div class="card table-card">

    <h2>
        Curves in selected folder
    </h2>

    <div class="table-wrapper">

        <table>

            <thead>

                <tr>

                    <th>Curve</th>

                    <th>kB (nN/µm)</th>

                    <th>E (MPa)</th>

                    <th>RMS (pN)</th>

                    <th>Filename</th>

                </tr>

            </thead>

            <tbody id="curveTableBody"></tbody>

        </table>

    </div>

</div>


<script>


// ============================================================
// GLOBAL STATE
// ============================================================

let manifest = [];

let currentFolderData = null;

let folderCache = {};


// ============================================================
// ELEMENTS
// ============================================================

const folderSelect =
    document.getElementById("folderSelect");

const curveSelect =
    document.getElementById("curveSelect");

const loading =
    document.getElementById("loading");

const xMin =
    document.getElementById("xMin");

const xMax =
    document.getElementById("xMax");

const curveCounter =
    document.getElementById("curveCounter");

const kbValue =
    document.getElementById("kbValue");

const eValue =
    document.getElementById("eValue");

const rmsValue =
    document.getElementById("rmsValue");

const jpkKcValue =
    document.getElementById("jpkKcValue");

const jpkSlopeValue =
    document.getElementById("jpkSlopeValue");

const jpkKbValue =
    document.getElementById("jpkKbValue");

const jpkEValue =
    document.getElementById("jpkEValue");

const jpkRmsValue =
    document.getElementById("jpkRmsValue");

const jpkContactValue =
    document.getElementById("jpkContactValue");

const jpkBaselineValue =
    document.getElementById("jpkBaselineValue");

const fitSlopeValue =
    document.getElementById("fitSlopeValue");

const fitKbValue =
    document.getElementById("fitKbValue");

const fitEValue =
    document.getElementById("fitEValue");

const fitRmsValue =
    document.getElementById("fitRmsValue");

const fitInterceptValue =
    document.getElementById("fitInterceptValue");

const fitStatusValue =
    document.getElementById("fitStatusValue");

const fitLinearSetting =
    document.getElementById("fitLinearSetting");

const fitElasticitySetting =
    document.getElementById("fitElasticitySetting");

const fitNuValue =
    document.getElementById("fitNuValue");

const fitAlphaValue =
    document.getElementById("fitAlphaValue");

const jpkTsvValue =
    document.getElementById("jpkTsvValue");

const curveValue =
    document.getElementById("curveValue");

const folderValue =
    document.getElementById("folderValue");

const excelValue =
    document.getElementById("excelValue");

const sheetValue =
    document.getElementById("sheetValue");

const filenameValue =
    document.getElementById("filenameValue");

const matchValue =
    document.getElementById("matchValue");

const tableBody =
    document.getElementById("curveTableBody");


// ============================================================
// FORMAT NUMBER
// ============================================================

function formatNumber(
    value,
    digits
) {

    if (
        value === null ||
        value === undefined ||
        Number.isNaN(Number(value))
    ) {

        return "N/A";
    }


    return Number(value).toFixed(
        digits
    );
}


// ============================================================
// LOAD MANIFEST
// ============================================================

async function loadManifest() {

    const response = await fetch(
        "data/manifest.json"
    );


    if (!response.ok) {

        throw new Error(
            "Could not load data/manifest.json"
        );
    }


    manifest = await response.json();


    folderSelect.innerHTML = "";


    manifest.forEach(entry => {

        const option =
            document.createElement(
                "option"
            );


        option.value =
            entry.folder;


        option.textContent =
            `${entry.folder} (${entry.count} curves)`;


        folderSelect.appendChild(
            option
        );

    });
}


// ============================================================
// LOAD ONE FOLDER JSON
// ============================================================

async function loadFolder(
    folderName
) {

    loading.textContent =
        "Loading folder data...";


    // --------------------------------------------------------
    // Use browser memory cache if already loaded
    // --------------------------------------------------------

    if (folderCache[folderName]) {

        currentFolderData =
            folderCache[folderName];

        loading.textContent = "";

        return;
    }


    const manifestEntry =
        manifest.find(
            item =>
                item.folder === folderName
        );


    if (!manifestEntry) {

        throw new Error(
            `Folder not found: ${folderName}`
        );
    }


    const response = await fetch(
        `data/${manifestEntry.file}`
    );


    if (!response.ok) {

        throw new Error(
            `Could not load ${manifestEntry.file}`
        );
    }


    const folderData =
        await response.json();


    folderCache[folderName] =
        folderData;


    currentFolderData =
        folderData;


    loading.textContent = "";
}


// ============================================================
// POPULATE CURVE DROPDOWN
// ============================================================

function populateCurveDropdown() {

    curveSelect.innerHTML = "";


    if (!currentFolderData) {

        return;
    }


    currentFolderData.curves.forEach(
        (curve, index) => {

            const option =
                document.createElement(
                    "option"
                );


            option.value =
                index;


            option.textContent =
                curve.curve_label;


            curveSelect.appendChild(
                option
            );

        }
    );


    if (
        currentFolderData.curves.length > 0
    ) {

        curveSelect.value = 0;
    }
}


// ============================================================
// BUILD CURVE TABLE
// ============================================================

function buildTable() {

    tableBody.innerHTML = "";


    if (!currentFolderData) {

        return;
    }


    currentFolderData.curves.forEach(
        (curve, index) => {

            const row =
                document.createElement(
                    "tr"
                );


            row.className =
                "data-row";


            row.dataset.index =
                index;


            row.innerHTML = `

                <td>
                    ${curve.curve_label}
                </td>

                <td>
                    ${formatNumber(curve.kB, 3)}
                </td>

                <td>
                    ${formatNumber(curve.E, 3)}
                </td>

                <td>
                    ${formatNumber(curve.RMS, 2)}
                </td>

                <td>
                    ${curve.filename}
                </td>

            `;


            row.addEventListener(
                "click",
                () => {

                    curveSelect.value =
                        index;

                    updatePlot();

                    window.scrollTo({
                        top: 0,
                        behavior: "smooth"
                    });

                }
            );


            tableBody.appendChild(
                row
            );

        }
    );
}


// ============================================================
// HIGHLIGHT SELECTED TABLE ROW
// ============================================================

function highlightSelectedRow(
    selectedIndex
) {

    document
        .querySelectorAll(
            ".data-row"
        )
        .forEach(row => {

            const rowIndex =
                Number(
                    row.dataset.index
                );


            if (
                rowIndex === selectedIndex
            ) {

                row.classList.add(
                    "selected"
                );

            } else {

                row.classList.remove(
                    "selected"
                );
            }

        });
}


// ============================================================
// UPDATE PLOT
// ============================================================

function updatePlot() {

    if (!currentFolderData) {

        return;
    }

    const index = Number(
        curveSelect.value
    );

    const curve =
        currentFolderData.curves[index];

    if (!curve) {

        return;
    }

    const fit = curve.fit || {};

    const jpk = curve.jpk || {};

    const fitOK = (
        fit.status === "OK"
    );

    // --------------------------------------------------------
    // Approach
    // --------------------------------------------------------

    const approachTrace = {
        x: curve.approach_x,
        y: curve.approach_y,
        type: "scatter",
        mode: "lines",
        name: "Approach",
        line: {
            color: "#1f77b4",
            width: 1.5
        }
    };

    // --------------------------------------------------------
    // Retract
    // --------------------------------------------------------

    const retractTrace = {
        x: curve.retract_x,
        y: curve.retract_y,
        type: "scatter",
        mode: "lines",
        name: "Retract",
        line: {
            color: "#ff7f0e",
            width: 1.5
        }
    };

    const traces = [
        approachTrace,
        retractTrace
    ];

    // --------------------------------------------------------
    // Linear fit
    // --------------------------------------------------------

    if (
        fitOK
        && Array.isArray(fit.linear_x_nm)
        && fit.linear_x_nm.length > 0
    ) {

        traces.push({
            x: fit.linear_x_nm,
            y: fit.linear_y_nN,
            type: "scatter",
            mode: "lines",
            name: "Linear fit",
            line: {
                color: "#2ca02c",
                width: 3
            },
            hovertemplate:
                "Linear fit<br>x=%{x:.3f} nm<br>y=%{y:.4f} nN<extra></extra>"
        });
    }

    // --------------------------------------------------------
    // Elasticity fit
    // --------------------------------------------------------

    if (
        fitOK
        && Array.isArray(fit.elasticity_x_nm)
        && fit.elasticity_x_nm.length > 0
    ) {

        traces.push({
            x: fit.elasticity_x_nm,
            y: fit.elasticity_y_nN,
            type: "scatter",
            mode: "lines",
            name: "Elasticity fit",
            line: {
                color: "#d62728",
                width: 3
            },
            hovertemplate:
                "Elasticity fit<br>x=%{x:.3f} nm<br>y=%{y:.4f} nN<extra></extra>"
        });
    }

    // --------------------------------------------------------
    // Fit-boundary and contact-point guide lines
    // --------------------------------------------------------

    const shapes = [];

    const boundary = Number(
        fit.force_boundary_nN
    );

    if (Number.isFinite(boundary)) {

        shapes.push({
            type: "line",
            xref: "paper",
            x0: 0,
            x1: 1,
            yref: "y",
            y0: boundary,
            y1: boundary,
            line: {
                color: "#777777",
                width: 1,
                dash: "dot"
            }
        });
    }

    const contactPoint = Number(
        jpk.contact_nm
    );

    if (Number.isFinite(contactPoint)) {

        shapes.push({
            type: "line",
            xref: "x",
            x0: contactPoint,
            x1: contactPoint,
            yref: "paper",
            y0: 0,
            y1: 1,
            line: {
                color: "#777777",
                width: 1,
                dash: "dot"
            }
        });
    }

    // --------------------------------------------------------
    // Plot layout
    // --------------------------------------------------------

    const layout = {
        title: {
            text:
                `${currentFolderData.folder} | ${curve.curve_label}`
        },

        xaxis: {
            title: "Indentation (nm)",
            range: [
                Number(xMin.value),
                Number(xMax.value)
            ],
            zeroline: true
        },

        yaxis: {
            title: "Force (nN)",
            zeroline: true,
            autorange: true
        },

        legend: {
            x: 0.01,
            y: 0.01,
            xanchor: "left",
            yanchor: "bottom",
            bgcolor: "rgba(255,255,255,0.80)",
            bordercolor: "#cccccc",
            borderwidth: 1
        },

        shapes: shapes,

        hovermode: "closest",

        margin: {
            l: 80,
            r: 30,
            t: 70,
            b: 70
        }
    };

    // --------------------------------------------------------
    // Plot controls
    // --------------------------------------------------------

    const config = {
        responsive: true,
        displaylogo: false,
        toImageButtonOptions: {
            format: "png",
            filename:
                `${currentFolderData.folder}_${curve.curve_label}`,
            scale: 2
        }
    };

    // --------------------------------------------------------
    // Draw graph
    // --------------------------------------------------------

    Plotly.react(
        "plot",
        traces,
        layout,
        config
    );

    // --------------------------------------------------------
    // Excel reference values
    // --------------------------------------------------------

    kbValue.textContent =
        formatNumber(
            curve.kB,
            3
        );

    eValue.textContent =
        formatNumber(
            curve.E,
            3
        );

    rmsValue.textContent =
        formatNumber(
            curve.RMS,
            2
        );

    // --------------------------------------------------------
    // Actual JPK values
    // --------------------------------------------------------

    jpkKcValue.textContent =
        formatNumber(
            jpk.kC_N_per_m,
            12
        );

    jpkSlopeValue.textContent =
        formatNumber(
            jpk.s_N_per_m,
            7
        );

    jpkKbValue.textContent =
        formatNumber(
            jpk.kB_nN_per_um,
            6
        );

    jpkEValue.textContent =
        formatNumber(
            jpk.E_MPa,
            6
        );

    jpkRmsValue.textContent =
        formatNumber(
            jpk.RMS_pN,
            6
        );

    jpkContactValue.textContent =
        formatNumber(
            jpk.contact_nm,
            6
        );

    jpkBaselineValue.textContent =
        formatNumber(
            jpk.baseline_pN,
            6
        );

    // --------------------------------------------------------
    // Fitted values
    // --------------------------------------------------------

    fitSlopeValue.textContent =
        formatNumber(
            fit.s_N_per_m,
            7
        );

    fitKbValue.textContent =
        formatNumber(
            fit.kB_nN_per_um,
            6
        );

    fitEValue.textContent =
        formatNumber(
            fit.E_MPa,
            6
        );

    fitRmsValue.textContent =
        formatNumber(
            fit.RMS_pN,
            6
        );

    fitInterceptValue.textContent =
        formatNumber(
            fit.linear_intercept_pN,
            6
        );

    fitStatusValue.textContent =
        fit.status || "N/A";

    if (fitOK) {

        fitStatusValue.className =
            "status-ok";

    } else {

        fitStatusValue.className =
            "status-error";
    }

    const forceBoundary = Number(
        fit.force_boundary_nN
    );

    if (Number.isFinite(forceBoundary)) {

        fitLinearSetting.textContent =
            `Linear: y ≥ ${forceBoundary.toFixed(3)} nN to curve maximum`;

        fitElasticitySetting.textContent =
            `Elasticity: y < ${forceBoundary.toFixed(3)} nN to −∞`;

    } else {

        fitLinearSetting.textContent =
            "Linear: N/A";

        fitElasticitySetting.textContent =
            "Elasticity: N/A";
    }

    fitNuValue.textContent =
        formatNumber(
            fit.poisson_ratio,
            2
        );

    fitAlphaValue.textContent =
        formatNumber(
            fit.pyramid_half_angle_deg,
            2
        );

    // --------------------------------------------------------
    // Metadata
    // --------------------------------------------------------

    curveValue.textContent =
        curve.curve_label;

    folderValue.textContent =
        currentFolderData.folder;

    excelValue.textContent =
        currentFolderData.excel_file
        ?? "N/A";

    sheetValue.textContent =
        currentFolderData.sheet
        ?? "N/A";

    jpkTsvValue.textContent =
        jpk.tsv_file
        ?? "N/A";

    filenameValue.textContent =
        curve.filename;

    matchValue.textContent =
        curve.match_status;

    if (
        curve.match_status === "OK"
    ) {

        matchValue.className =
            "status-ok";

    } else {

        matchValue.className =
            "status-error";
    }

    // --------------------------------------------------------
    // Curve counter
    // --------------------------------------------------------

    curveCounter.textContent =
        `Curve ${index + 1} of `
        + `${currentFolderData.curves.length}`;

    // --------------------------------------------------------
    // Highlight table row
    // --------------------------------------------------------

    highlightSelectedRow(
        index
    );
}


// ============================================================
// FOLDER CHANGED
// ============================================================

async function folderChanged() {

    try {

        await loadFolder(
            folderSelect.value
        );


        populateCurveDropdown();


        buildTable();


        updatePlot();

    } catch (error) {

        loading.textContent =
            error.message;

        console.error(
            error
        );
    }
}


// ============================================================
// PREVIOUS CURVE
// ============================================================

function previousCurve() {

    let index =
        Number(
            curveSelect.value
        );


    if (index > 0) {

        index -= 1;

        curveSelect.value =
            index;

        updatePlot();
    }
}


// ============================================================
// NEXT CURVE
// ============================================================

function nextCurve() {

    if (!currentFolderData) {

        return;
    }


    let index =
        Number(
            curveSelect.value
        );


    if (
        index <
        currentFolderData.curves.length - 1
    ) {

        index += 1;

        curveSelect.value =
            index;

        updatePlot();
    }
}


// ============================================================
// EVENT LISTENERS
// ============================================================

folderSelect.addEventListener(

    "change",

    folderChanged
);


curveSelect.addEventListener(

    "change",

    updatePlot
);


document
    .getElementById(
        "previousButton"
    )
    .addEventListener(

        "click",

        previousCurve
    );


document
    .getElementById(
        "nextButton"
    )
    .addEventListener(

        "click",

        nextCurve
    );


document
    .getElementById(
        "applyAxis"
    )
    .addEventListener(

        "click",

        updatePlot
    );


// ============================================================
// START DASHBOARD
// ============================================================

async function startDashboard() {

    try {

        loading.textContent =
            "Loading dashboard...";


        await loadManifest();


        if (
            manifest.length === 0
        ) {

            loading.textContent =
                "No AFM folders found.";

            return;
        }


        folderSelect.value =
            manifest[0].folder;


        await folderChanged();


        loading.textContent = "";

    } catch (error) {

        loading.textContent =
            error.message;

        console.error(
            error
        );
    }
}


startDashboard();


</script>


</body>

</html>
"""


# ============================================================
# INSERT DEFAULT X RANGE
# ============================================================

html = (

    html

    .replace(
        "__X_MIN__",
        str(DEFAULT_X_MIN)
    )

    .replace(
        "__X_MAX__",
        str(DEFAULT_X_MAX)
    )
)


# ============================================================
# SAVE INDEX.HTML
# ============================================================

INDEX_FILE.write_text(

    html,

    encoding="utf-8"
)


# ============================================================
# REPORT RESULTS
# ============================================================

json_files = list(
    DATA_FOLDER.glob("*.json")
)


total_json_size = sum(

    file.stat().st_size

    for file
    in json_files
)


largest_json = max(

    json_files,

    key=lambda file:
        file.stat().st_size
)


print()

print(
    "=========================================="
)

print(
    "GitHub Pages dashboard created"
)

print(
    "=========================================="
)

print()

print(
    "index.html:"
)

print(
    INDEX_FILE.resolve()
)

print()

print(
    "Data folder:"
)

print(
    DATA_FOLDER.resolve()
)

print()

print(
    "AFM folders:",
    len(manifest)
)

print(
    "Total curves:",
    total_curves
)

print()

print(
    "Total JSON size:",
    round(
        total_json_size
        /
        1024
        /
        1024,
        2
    ),
    "MB"
)

print(
    "Largest JSON file:",
    largest_json.name
)

print(
    "Largest JSON size:",
    round(
        largest_json.stat().st_size
        /
        1024
        /
        1024,
        2
    ),
    "MB"
)

print()

print(
    "Generation complete."
)


Processing: 20_cell1part1 (93 curves)
Excel: 20.xlsx
Sheet: cell1part1
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell1part2 (83 curves)
Excel: 20.xlsx
Sheet: cell1part2
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell2part1 (61 curves)
Excel: 20.xlsx
Sheet: cell2part1
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell2part2 (83 curves)
Excel: 20.xlsx
Sheet: cell2part2
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell3part1 (36 curves)
Excel: 20.xlsx
Sheet: cell3part1
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell3part2 (47 curves)
Excel: 20.xlsx
Sheet: cell3part2
Excel status: OK
JPK TSV status: No processed JPK TSV found
JPK TSV files: []

Processing: 20_cell3part3 (25 curves)
Excel: 20.xlsx
Sheet: cell3part3
Excel status: OK
JPK TSV sta